# 02 검색 기준선과 원인 분리

목표: 단어/문자 TF-IDF와 버전 필터를 따로 비교합니다. TF-IDF는 Term Frequency–Inverse Document Frequency입니다.

**준비:** 저장소 터미널에서 python -m pip install -r requirements/week11-offline.txt 실행.
VS Code 커널에서 저장소 .venv의 **Python 3.11** 선택 후 위에서 아래로 실행합니다.
각 파일은 다른 Notebook의 변수를 요구하지 않습니다. GPU·Ollama·API 키는 필요 없습니다.

**데이터:** 교수 작성 가상 쇼핑 정책입니다. 실제 기업 규정이나 고객 기록이 아닙니다.
핵심 단계는 수업에서 함께 실행하고 확장 실험은 복습용입니다. [실습 안내](../README.md).


In [ ]:
from pathlib import Path
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week11/lab/evaluation.py").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("저장소 안에서 Notebook을 실행하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("저장소 .venv의 Python 3.11 커널을 선택하세요.")
import pandas as pd
from IPython.display import display
from week11.lab.evaluation import load_data, Config, Retriever, evaluate, summarize

docs, questions = load_data()
dev = [q for q in questions if q["split"] == "dev"]
test = [q for q in questions if q["split"] == "test"]
print("환경 준비 완료", sys.version.split()[0], "문서", len(docs), "질문", len(questions))

## 1. 단어 단위 기준선

문서 내 빈도와 집합에서의 희소성을 결합합니다. 의미를 이해하는 LLM은 아닙니다.

In [ ]:
retriever = Retriever(docs, Config("word-all"))
display(pd.DataFrame(retriever.search("단순 변심 반품 배송비는 얼마인가요?")))

**확인과 해석**

점수는 정답 확률이 아닙니다. 유사도와 정책 유효성을 구분합니다.

## 2. 벡터 크기

문서만 fit하고 질문은 transform합니다.

In [ ]:
print("문서 × 특징:", retriever.matrix.shape)
v = retriever.vectorizer.transform(["반품 배송비"])
print(retriever.vectorizer.get_feature_names_out()[v.indices])

**확인과 해석**

word 토큰화는 한국어 형태소 분석기가 아닙니다.

## 3. 겹치는 단어가 없는 질문

0점 문서 반환은 가짜 검색 성공을 만들 수 있습니다.

In [ ]:
print(retriever.search("zzzzzzzz"))
assert retriever.search("zzzzzzzz") == []

**확인과 해석**

빈 검색 결과와 서버 장애는 다릅니다.

## 4. 문자 단위 검색

2~4그램은 부분 문자열을 공유하지만 동의어 의미까지 해결하지는 못합니다.

In [ ]:
char = Retriever(docs, Config("char-current", "char", True))
for query in ["반품 배송비", "반품배송비", "돌려보내면 택배비 얼마죠?"]:
    print(query)
    display(pd.DataFrame(char.search(query))[["id", "title", "score"]])

**확인과 해석**

문자 TF-IDF는 신경망 임베딩이 아닙니다.

## 5. 한 요소씩 비교

필터와 알고리즘을 동시에 바꾸면 원인이 불명확합니다.

In [ ]:
configs = [
    Config("word-all"),
    Config("word-current", "word", True),
    Config("char-current", "char", True),
]
comparison = pd.DataFrame([{"name": c.name, **summarize(evaluate(docs, dev, c))} for c in configs])
display(comparison[["name", "recall", "mrr", "archive_in_topk", "false_accept"]])

**확인과 해석**

dev에서 원인을 탐색하고 test로 설정을 고르지 않습니다.

## 6. 동점 처리

동점 규칙도 재현 조건입니다.

In [ ]:
tiny = [dict(id=x, title="반품", text="반품", status="current") for x in ["B", "A"]]
print(Retriever(tiny, Config("tie")).search("반품"))

**확인과 해석**

문서 ID 오름차순으로 동점을 해소합니다.

## 확장 실험과 프로젝트 적용

1. analyzer와 current_only 효과를 분리합니다.
2. 실제 임베딩 검색기로 바꿀 때 id/score/text/status 계약을 유지합니다.
3. 모델·정규화·거리 함수·인덱스 버전을 기록합니다.

**실행 실패 시:** ModuleNotFoundError는 커널과 설치 터미널이 같은 .venv인지 확인합니다.
ROOT 오류는 저장소 안에서 파일을 열면 해결됩니다. 변수가 섞였다면 Restart Kernel 후 Run All.
예외 실험은 try/except로 설명하므로 정상 실행에서는 빨간 오류로 끝나지 않습니다.

**기록:** 변경 설정 / 예상 / 실제 관찰 / 실패 질문 ID / 다음 행동을 5줄로 적습니다.
개인 프로젝트 원문·연락처·API 키는 공개 GitHub에 올리지 않습니다.
